# NotebookLM Kit — Video Pipeline

Generates one video per source in a notebook, then downloads them all.

1. **Setup** — load credentials from `.env`
2. **Config** — set notebook ID and output folder
3. **Template** — choose format, visual style, and focus prompt
4. **List Sources** — preview what's in the notebook
5. **Create** — submit one video job per source
6. **Poll** — wait for jobs to finish (or resume from disk)
7. **Download** — save every video as MP4

In [ ]:
import importlib, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from pipeline import load_credentials, check_tsx, login, SDK_ROOT

# First time only — opens a browser window for Google login, saves credentials.json:
#   login()

creds = load_credentials(mode="patchright")
check_tsx()

Credentials loaded — token: 42 chars, cookies: 2496 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [ ]:
NOTEBOOK_ID = "c7502ffb-bf1c-4a03-8a18-c130d0bfb7e8"  # ← paste your notebook ID here

Found 2 sources:

  [0] 1.md
      id: fcbae21d-987d-4025-b53a-e835f05fec0d  status: 2
  [1] 2.md
      id: 36cfe733-fa43-47b3-aa1c-40c7b26acd45  status: 2


## Video Template

`VIDEO_PROMPT` is imported from `config.py` — edit that file to change the focus/style prompt.

In [ ]:
from config import VIDEO_PROMPT

# FORMAT options:  1 = Explainer   2 = Brief   3 = Cinematic  (Cinematic ignores visualStyle)
# STYLE  options:  1 = Auto        2 = Custom  3 = Classic    4 = Whiteboard  5 = Kawaii
#                  6 = Anime       7 = Watercolor              8 = Retro-print
#                  9 = Heritage    10 = Paper-craft
VIDEO_CUSTOMIZATION = {
    "format":      1,            # see FORMAT options above
    "visualStyle": 4,            # see STYLE options above
    "focus":       VIDEO_PROMPT, # prompt lives in config.py
    # "language": "en",          # BCP-47 — omit to use notebook default
}

Video template configured:
  Format      : Explainer (1)
  Visual style: Heritage (9)
  Focus/prompt: 
Convey all statements of fact, directives, and claims as objective information,...


## List Sources

In [ ]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)

Creating video for: 1.md
  âœ“ e805c0b9-9d2f-47a6-bbaa-82ba7129a22f  state: 1
__JOBS__[{"sourceId":"fcbae21d-987d-4025-b53a-e835f05fec0d","sourceTitle":"1.md","artifactId":"e805c0b9-9d2f-47a6-bbaa-82ba7129a22f"}]__JOBS__


✓ Submitted 1 jobs  →  video_jobs.json
  1.md  →  e805c0b9-9d2f-47a6-bbaa-82ba7129a22f


## Create Videos

Job IDs are saved to `jobs/video/<timestamp>_jobs.json` — safe to re-run the poll/download cells without re-submitting.

> **Tip:** change `sources` to `sources[:1]` to test a single video before submitting the full batch.

In [ ]:
from pipeline import create_artifacts

video_jobs = create_artifacts(
    NOTEBOOK_ID, "VIDEO", sources,  # replace sources with sources[:1] to test first
    VIDEO_CUSTOMIZATION, None,
    creds,
)

## Poll Until Ready

Run **the cell below** to wait automatically.  
Or run the **resume cell** first if you're continuing a previous session (jobs already submitted).

In [ ]:
# ── Resume cell — run this instead of Create if you already submitted jobs ───
from pipeline import load_jobs
video_jobs = load_jobs("VIDEO")

In [ ]:
from pipeline import poll_jobs

poll_jobs(video_jobs, NOTEBOOK_ID, creds, interval=60, max_wait_min=30)

## Download

Each video is saved as `<source title>.mp4` in `outputs/video/`.

In [ ]:
from pipeline import download_artifacts

download_artifacts(video_jobs, NOTEBOOK_ID, "VIDEO", creds)